# Generate Executive Summary DOCX

Builds a formatted Word document summarising the CLL epidemiology pipeline status.  
Covers: what was built, the hybrid automation pipeline, validation framework,  
harmonization, forecasting approaches, and proposed next steps.

**Output:** `Executive_Summary_CLL.docx` (or path set in Cell 2)  
**Requires:** `python-docx` (`pip install python-docx`)

In [ ]:
# Cell 1 — Imports
from __future__ import annotations
from datetime import date
from pathlib import Path
try:
    from docx import Document
    from docx.shared import Inches
    print('python-docx OK')
except ImportError:
    print('ERROR: python-docx not installed. Run: pip install python-docx')

In [ ]:
# Cell 2 — Configuration
OUTPUT_PATH = Path().resolve().parent / "Executive_Summary_CLL.docx"
DOC_DATE    = date.today().isoformat()   # Override: e.g. "2026-04-29"

print(f'Output: {OUTPUT_PATH}')
print(f'Date:   {DOC_DATE}')

In [ ]:
# Cell 3 — Document builder

def _bullets(doc, items):
    for item in items:
        doc.add_paragraph(item, style="List Bullet")


def build_document(doc_date: str):
    doc = Document()
    s = doc.sections[0]
    s.top_margin = s.bottom_margin = Inches(0.5)
    s.left_margin = s.right_margin = Inches(0.5)

    doc.add_heading("Executive summary — CLL epidemiology and pipeline (status update)", level=1)
    doc.add_paragraph(f"Date: {doc_date}")

    doc.add_heading("Executive summary", level=2)
    doc.add_paragraph(
        "We have completed the extraction and structuring of core CLL epidemiology, and we have implemented a "
        "hybrid automation data pipeline to consolidate evidence from authoritative sources while maintaining data "
        "quality and full traceability. In parallel, we conducted a focused literature review to identify the "
        "clinical flow parameters required for patient journey modeling, and we established forecasting methods "
        "to project epidemiology over time under conservative and growth scenarios."
    )

    doc.add_heading("What has been completed", level=2)
    _bullets(doc, [
        "Core CLL epidemiology extracted and structured from SEER (incidence, prevalence, survival, mortality) with stratifications.",
        "Master dataset created with standardized fields and full traceability (source tagging and data type classification).",
        "Literature review completed for patient journey parameters; KPI logic defined (drug ready, treated, LOT segmentation).",
        "Initial patient flow generated from total prevalence through 1L, 2L, and 3L plus pools.",
        "Forecasting implemented (adapted APC + log linear); US Census population integrated.",
        "Dynamic one-stop Excel v6 deliverable created with pipeline auto-population via excel_updater.py.",
        "Semantic metric matching, URL resolver, confidence scoring, and metric clustering added to pipeline.",
    ])

    doc.add_heading("Hybrid automation data pipeline", level=2)
    doc.add_paragraph(
        "A hybrid pipeline addresses the challenge of collecting trusted oncology epidemiology across sources "
        "with varying access models. Three modes: web scraping where allowed, structured file ingestion for "
        "portal-only sources, and scheduled manual extraction for desktop tools."
    )
    _bullets(doc, [
        "Source list defined across gold/silver/bronze tiers (registries, guidelines, journals, APIs).",
        "Standardized evidence schema: metric, definition, population, year, geography, value, citation, URL, tier.",
        "Reproducible execution via CLI, notebook, and batch runner (run_all_indications.py).",
        "Exports: evidence tables, KPI scorecards, tool-ready tables, InsightACE format, cluster-tagged evidence.",
    ])

    doc.add_heading("Validation framework", level=2)
    _bullets(doc, [
        "Cross-source checks: automatic comparison of key metrics; flags when sources disagree.",
        "CLL business rules: mortality cannot exceed incidence; survival within expected modern ranges.",
        "Confidence scoring: 0–100 rule-based score (tier + extraction + completeness + recency + URL).",
        "Conflict detection and reconciliation output with ranges and recommended values.",
    ])

    doc.add_heading("Forecasting", level=2)
    _bullets(doc, [
        "Conservative scenario: adapted APC model with trend tapering.",
        "Growth scenario: log linear model with sustained growth.",
        "Population: US Census projections; UN-based sensitivity planned.",
    ])

    doc.add_heading("Ready deliverables", level=2)
    _bullets(doc, [
        "Excel v6 one-stop workbook (Evidence Import + dynamic IFERROR/INDEX/MATCH formulas in all 6 Forecast sheets).",
        "Evidence tables, KPI scorecard, conflicts, reconciliation, white-space coverage summary.",
        "Forecast outputs (conservative + growth), InsightACE epidemiology format.",
        "Streamlit app at https://epi-data-tool.streamlit.app/; Jupyter notebook; batch CLI.",
        "Power BI and Tableau dashboard packages with shared bi_data/ export layer.",
    ])

    doc.add_heading("Proposed next steps", level=2)
    _bullets(doc, [
        "Validate key modeling assumptions (treatment uptake, LOT transitions, eligibility definitions).",
        "Agree on refresh cadence and document in a lightweight run book.",
        "Add UN population sensitivity runs vs Census-based forecasts.",
        "Test excel_updater.py end-to-end; wire into run_all_indications.py --update-excel flag.",
        "Commit session 4 pipeline changes to GitHub and redeploy Streamlit.",
    ])

    return doc


print('Builder function defined')

In [ ]:
# Cell 4 — Build and save
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
doc = build_document(doc_date=DOC_DATE)
doc.save(OUTPUT_PATH)
print(f'Saved: {OUTPUT_PATH}  ({OUTPUT_PATH.stat().st_size:,} bytes)')